In [1]:
from pathlib import Path
import numpy as np
from PIL import Image
from scipy.io import loadmat

In [7]:
data_path = Path("E:/course_file/26fall/18-794/18-794_project/data/Penn_Action")


def check_video(video_id):
    video_id = str(video_id).zfill(4)

    frame_dir = data_path / "frames" / video_id
    label_path = data_path / "labels" / f"{video_id}.mat"

    label = loadmat(label_path)

    train_value = int(label["train"].item())

    if train_value == 1:
        split = "train"
    elif train_value == -1:
        split = "test"
    else:
        raise ValueError(f"Unexpected train value: {train_value} in {label_path}")

    x = np.asarray(label["x"])
    y = np.asarray(label["y"])

    if x.ndim != 2 or x.shape != y.shape or x.shape[1] != 13:
        raise ValueError(f"Unexpected pose shape: x={x.shape}, y={y.shape}")

    pose_frames = x.shape[0]
    expected_frames = int(np.asarray(label.get("nframes", pose_frames)).item())

    frame_files = []

    for p in frame_dir.iterdir():
        if p.is_file() and p.suffix.lower() in {".jpg"}:
            frame_files.append(p)

    if any(not p.stem.isdigit() for p in frame_files):
        raise ValueError("Frame filenames must have numeric stems")

    frame_files.sort()
    
    frame_ids = []
    for p in frame_files:
        frame_id = int(p.stem)
        frame_ids.append(frame_id)

    expected_ids = set(range(1, expected_frames + 1))
    missing_ids = sorted(expected_ids - set(frame_ids))

    with Image.open(frame_files[0]) as image:
        width, height = image.size

    return {
        "video_id": video_id,
        "split": split,
        "rgb_frames": len(frame_files),
        "pose_frames": pose_frames,
        "frames_match": len(frame_files) == pose_frames,
        "has_missing_frames": bool(missing_ids),
        "missing_frame_ids": missing_ids,
        "width": width,
        "height": height,
        "unexpected_frame_ids": sorted(set(frame_ids) - expected_ids),
        "has_duplicate_frame_ids": len(frame_ids) != len(set(frame_ids)),
    }

In [8]:
print(check_video("0001"))

{'video_id': '0001', 'split': 'test', 'rgb_frames': 151, 'pose_frames': 151, 'frames_match': True, 'has_missing_frames': False, 'missing_frame_ids': [], 'width': 480, 'height': 360, 'unexpected_frame_ids': [], 'has_duplicate_frame_ids': False}


In [9]:
import pandas as pd
results = []

for label_path in sorted((data_path / "labels").glob("*.mat")):
    video_id = label_path.stem
    results.append(check_video(video_id))

df = pd.DataFrame(results)

output_path = Path("penn_data_validation.csv")
df.to_csv(output_path, index=False)

df.head(5)

,video_id,split,rgb_frames,pose_frames,frames_match,has_missing_frames,missing_frame_ids,width,height,unexpected_frame_ids,has_duplicate_frame_ids
0,0001,test,151,151,True,False,[],480,360,[],False
1,0002,train,80,80,True,False,[],480,270,[],False
2,0003,train,85,85,True,False,[],480,270,[],False
3,0004,train,82,82,True,False,[],480,270,[],False
4,0005,train,48,48,True,False,[],480,270,[],False


In [10]:
df["split"].value_counts()

split
train    1258
test     1068
Name: count, dtype: int64

In [11]:
df["frames_match"].value_counts()

frames_match
True    2326
Name: count, dtype: int64

In [13]:
df["has_duplicate_frame_ids"].value_counts()

has_duplicate_frame_ids
False    2326
Name: count, dtype: int64

In [12]:
df["rgb_frames"].describe()

count    2326.000000
mean       70.438951
std        54.799477
min        18.000000
25%        43.000000
50%        56.000000
75%        78.000000
max       663.000000
Name: rgb_frames, dtype: float64

This dataset has no missing values or missing frames. All pose frame and rgb frame can be matched.